# Initial Verification
___
A RDS database has been deployed and populated as part of the backend deployment to demonstrate the interaction of the chatbot with a pre-existent database.  

The main porpuse of this notebooks is get you familized with the data that has been pre-loaded in the database for your further chatbot interaction test.


![Architecture Diagram](../images/infra3.png)

## Initial Setup and Imports
### First cell includes commented-out pip installations for required packages:
___
- langgraph
- psycopg2-binary
- python-dotenv

In [ ]:
# %%capture --no-stderr
# !pip install -U langgraph
# !pip install psycopg2-binary==2.9.9
# !pip install python-dotenv

In [1]:
import os
import sys
import json
import boto3
from botocore.exceptions import ClientError
from dotenv import load_dotenv

# Setup Path and Environment
Add backend directory to system path for module imports

In [2]:
# Get the absolute path to the backend directory
current_dir = os.getcwd()  # Get current working directory
backend_path = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(backend_path)

# Load environment variables from .env file
load_dotenv()

True

# AWS Secrets Manager Function
Function to retrieve database credentials from AWS Secrets Manager.
- `secret_name` can be extracted from the databse_stack deployed.

In [3]:
def get_secret():
    secret_name = "ChatbotDatabase*******"
    region_name = os.getenv('AWS_REGION')

    # Create a Secrets Manager client
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        raise e

    # Parse the JSON string into a dictionary
    secret_dict = json.loads(get_secret_value_response['SecretString'])
    
    return secret_dict


# Database Operations
___
Connect to PostgreSQL database and retrieve data
First query: Retrieve all orders
- Connects to database using credentials from Secrets Manager
- Executes SELECT query on orders table
- Prints all orders with their details (id, customer_id, product, quantity, price, status)


In [6]:
import psycopg2

# Database connection parameters from the secret
db_params = {
    'host': get_secret()['host'],
    'database': get_secret()['dbname'],
    'user':  get_secret()['username'],
    'password': get_secret()['password'],
    'port': get_secret()['port']
}

# Connect to the database
conn = psycopg2.connect(**db_params)
cursor = conn.cursor()

# Query to get all data from orders table
cursor.execute("SELECT * FROM orders")

# Fetch and print all orders
orders = cursor.fetchall()
for order in orders:
    order_dict = dict(
        zip(
            [
                "id",
                "customer_id",
                "product",
                "quantity",
                "price",
                "status",
            ],
            order,
        )
    )
    print(order_dict)

cursor.close()
conn.close()

{'id': '1', 'customer_id': '1', 'product': 'Sample Product', 'quantity': 1, 'price': Decimal('99.99'), 'status': 'pending'}
{'id': '24601', 'customer_id': '123456', 'product': 'Wireless Headphones', 'quantity': 1, 'price': Decimal('79.99'), 'status': 'Shipped'}
{'id': '13579', 'customer_id': '123456', 'product': 'Smartphone Case', 'quantity': 2, 'price': Decimal('19.99'), 'status': 'Processing'}
{'id': '97531', 'customer_id': '234567', 'product': 'Bluetooth Speaker', 'quantity': 1, 'price': Decimal('49.99'), 'status': 'Shipped'}
{'id': '86420', 'customer_id': '345678', 'product': 'Fitness Tracker', 'quantity': 1, 'price': Decimal('129.99'), 'status': 'Delivered'}
{'id': '54321', 'customer_id': '456789', 'product': 'Laptop Sleeve', 'quantity': 3, 'price': Decimal('24.99'), 'status': 'Shipped'}
{'id': '19283', 'customer_id': '567890', 'product': 'Wireless Mouse', 'quantity': 1, 'price': Decimal('34.99'), 'status': 'Processing'}
{'id': '74651', 'customer_id': '678901', 'product': 'Gaming 

___
# Second query: Retrieve all customers
- Uses same database connection
- Executes SELECT query on customers table
- Prints all customer information (id, name, email, phone, username)

In [7]:
import psycopg2

# Database connection parameters from the secret
db_params = {
    'host': get_secret()['host'],
    'database': get_secret()['dbname'],
    'user':  get_secret()['username'],
    'password': get_secret()['password'],
    'port': get_secret()['port']
}

# Connect to the database
conn = psycopg2.connect(**db_params)
cursor = conn.cursor()

# Query to get all data from customers table
cursor.execute("SELECT * FROM customers")

# Fetch and print all customers
customers = cursor.fetchall()
for customer in customers:
    customer_dict = dict(
        zip(
            [
                "id",
                "name",
                "email",
                "phone",
                "username"
            ],
            customer,
        )
    )
    print(customer_dict)

cursor.close()
conn.close()

{'id': '1', 'name': 'John Doe', 'email': 'john@example.com', 'phone': '123-456-7890', 'username': 'johndoe'}
{'id': '123456', 'name': 'Alejandro Rosalez', 'email': 'alejandro_rosalez@example.com', 'phone': '619-555-3782', 'username': 'alejandro_rosalez'}
{'id': '234567', 'name': 'Akua Mansa', 'email': 'akua_mansa@example.com', 'phone': '202-555-9271', 'username': 'akua_mansa'}
{'id': '345678', 'name': 'Ana Carolina Silva', 'email': 'anacarolina_silva@example.com', 'phone': '808-555-6149', 'username': 'anacarolina_silva'}
{'id': '456789', 'name': 'Arnav Desai', 'email': 'arnav_desai@example.com', 'phone': '312-555-8204', 'username': 'arnav_desai'}
{'id': '567890', 'name': 'Carlos Salazar', 'email': 'carlos_salazar@example.com', 'phone': '718-555-4965', 'username': 'carlos_salazar'}
{'id': '678901', 'name': 'Diego Ramirez', 'email': 'diego_ramirez@example.com', 'phone': '305-555-7183', 'username': 'diego_ramirez'}
{'id': '789012', 'name': 'Efua Owusu', 'email': 'efua_owusu@example.com', 